# 10. Carga masiva de documentos diarios en MongoDB

## Objetivo

Automatizar la construcción e inserción de los documentos diarios correspondientes a todas las fechas disponibles en PostgreSQL.

Antes de ejecutar la carga completa, el proceso se valida con un subconjunto reducido de fechas. Cada día se procesa de forma independiente para evitar que un error puntual detenga la ejecución completa.

El flujo reutiliza las funciones previamente validadas para:

- cargar los datos diarios desde PostgreSQL;
- construir el documento de resumen;
- generar las gráficas;
- añadir sus metadatos;
- insertar o actualizar el documento mediante `upsert`;
- registrar los días procesados correctamente y los días fallidos.

In [ ]:
from datetime import datetime
from time import perf_counter

import pandas as pd

from src.database.connection import get_database_engine
from src.database.load_available_dates import load_available_dates

from src.mongodb.connection import get_mongodb_database
from src.mongodb.load_documents import (
    create_daily_documents_index,
)

from src.mongodb.process_daily_document import process_daily_document

In [3]:
dataset_version = "v3"
collection_name = "daily_summaries"

output_directory = (
    project_root
    / "outputs"
    / "daily_plots"
)

test_limit = 31

In [4]:
postgres_engine = get_database_engine()

mongodb_database, mongodb_client = get_mongodb_database()

daily_collection = mongodb_database[collection_name]

index_name = create_daily_documents_index(
    daily_collection
)

print("Conexiones establecidas correctamente.")
print(f"Base de datos MongoDB: {mongodb_database.name}")
print(f"Colección: {daily_collection.name}")
print(f"Índice disponible: {index_name}")

Conexiones establecidas correctamente.
Base de datos MongoDB: solar_irradiance_db
Colección: daily_summaries
Índice disponible: uq_daily_document_date_dataset_version


In [5]:
available_dates = load_available_dates(
    postgres_engine
)

dates_to_process = available_dates

In [6]:
successful_dates = []
failed_dates = []

start_time = perf_counter()

total_dates = len(dates_to_process)

for position, date in enumerate(dates_to_process, start=1):
    try:
        result = process_daily_document(
            date=date,
            postgres_engine=postgres_engine,
            mongodb_collection=daily_collection,
            output_directory=output_directory,
            dataset_version=dataset_version,
        )

        successful_dates.append(result)

        print(
            f"[{position}/{total_dates}] "
            f"{date}: procesada correctamente "
            f"({result['registros']:,} registros)."
        )

    except Exception as exc:
        failed_dates.append(
            {
                "fecha": date,
                "error": str(exc),
            }
        )

        print(
            f"[{position}/{total_dates}] "
            f"{date}: error durante el procesamiento: {exc}"
        )

elapsed_time = perf_counter() - start_time

[1/731] 2023-01-01: procesada correctamente (1,440 registros).
[2/731] 2023-01-02: procesada correctamente (1,440 registros).
[3/731] 2023-01-03: procesada correctamente (1,440 registros).
[4/731] 2023-01-04: procesada correctamente (1,440 registros).
[5/731] 2023-01-05: procesada correctamente (1,440 registros).
[6/731] 2023-01-06: procesada correctamente (1,440 registros).
[7/731] 2023-01-07: procesada correctamente (1,440 registros).
[8/731] 2023-01-08: procesada correctamente (1,440 registros).
[9/731] 2023-01-09: procesada correctamente (1,440 registros).
[10/731] 2023-01-10: procesada correctamente (1,440 registros).
[11/731] 2023-01-11: procesada correctamente (1,440 registros).
[12/731] 2023-01-12: procesada correctamente (1,440 registros).
[13/731] 2023-01-13: procesada correctamente (1,440 registros).
[14/731] 2023-01-14: procesada correctamente (1,440 registros).
[15/731] 2023-01-15: procesada correctamente (1,440 registros).
[16/731] 2023-01-16: procesada correctamente (1,4

In [7]:
print("\nResumen de la carga completa")
print("-" * 40)
print(f"Fechas disponibles: {total_dates:,}")
print(f"Fechas procesadas correctamente: {len(successful_dates):,}")
print(f"Fechas con error: {len(failed_dates):,}")
print(f"Tiempo total: {elapsed_time / 60:.2f} minutos")


Resumen de la carga completa
----------------------------------------
Fechas disponibles: 731
Fechas procesadas correctamente: 731
Fechas con error: 0
Tiempo total: 38.70 minutos


In [8]:
processed_dates = [
    result["fecha"]
    for result in successful_dates
]

processed_datetimes = [
    datetime.strptime(date, "%Y-%m-%d")
    for date in processed_dates
]

stored_documents = daily_collection.count_documents(
    {
        "fecha": {
            "$in": processed_datetimes
        },
        "dataset.version": dataset_version,
    }
)

print(f"Documentos encontrados en MongoDB: {stored_documents:,}")
print(f"Documentos esperados: {len(successful_dates):,}")

assert stored_documents == len(successful_dates)

print(
    "Los documentos almacenados coinciden con "
    "las fechas procesadas correctamente."
)

Documentos encontrados en MongoDB: 731
Documentos esperados: 731
Los documentos almacenados coinciden con las fechas procesadas correctamente.


In [9]:
first_document = daily_collection.find_one(
    {
        "dataset.version": dataset_version,
    },
    sort=[("fecha", 1)],
)

last_document = daily_collection.find_one(
    {
        "dataset.version": dataset_version,
    },
    sort=[("fecha", -1)],
)

if first_document is None or last_document is None:
    raise ValueError(
        "No se han encontrado documentos para la versión indicada."
    )

print(f"Primera fecha en MongoDB: {first_document['fecha']}")
print(f"Última fecha en MongoDB: {last_document['fecha']}")

Primera fecha en MongoDB: 2023-01-01 00:00:00
Última fecha en MongoDB: 2024-12-31 00:00:00


In [10]:
if failed_dates:
    failed_dates_df = pd.DataFrame(failed_dates)
    display(failed_dates_df)
else:
    print("No se han producido errores durante la carga completa.")

No se han producido errores durante la carga completa.


In [11]:
logs_directory = (
    project_root
    / "outputs"
    / "mongodb_logs"
)

logs_directory.mkdir(
    parents=True,
    exist_ok=True,
)

successful_dates_df = pd.DataFrame(successful_dates)
failed_dates_df = pd.DataFrame(failed_dates)

successful_dates_df.to_csv(
    logs_directory
    / "mongodb_bulk_load_success.csv",
    index=False,
)

failed_dates_df.to_csv(
    logs_directory
    / "mongodb_bulk_load_errors.csv",
    index=False,
)

print(f"Logs guardados en: {logs_directory}")

Logs guardados en: c:\Users\zacar\Desktop\Master IEBS DS & Big Data\TFM_IEBS\outputs\mongodb_logs


In [12]:
mongodb_client.close()

print("Conexión con MongoDB cerrada correctamente.")

Conexión con MongoDB cerrada correctamente.
